# Paper 5 — CA-IEDI actual DMM and trust gate

Uses the ambiguity/cold-start DMM, bounded four-agent runtime, acoustic-evidence interface, and fail-closed provenance state machine. The default provenance/authorizer are explicit offline fixtures, not a deployed DAO.


In [ ]:
from pathlib import Path
import os
import sys

search_roots = (Path.cwd(), *Path.cwd().parents, Path("/content/iedi-mas"))
ROOT = next((path for path in search_roots if (path / "src" / "iedi").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Repository not found. Clone it and install with: pip install -e .[gemini]")
sys.path.insert(0, str(ROOT / "src"))

from iedi.codebook import Codebook
from iedi.providers import GoogleGenAIProvider, OfflineFixtureProvider
from iedi.pipeline import build_pipeline
from iedi.schemas import InterpretationRequest

codebook = Codebook.from_json(ROOT / "data" / "codebook.demo.json")
# OfflineFixtureProvider only echoes reviewed evidence; it is never empirical evidence.
# Set IEDI_LIVE_GEMINI=1 and GEMINI_API_KEY to exercise the real 2.5 Flash/Pro adapter.
LIVE_GEMINI = os.getenv("IEDI_LIVE_GEMINI") == "1"
provider = GoogleGenAIProvider() if LIVE_GEMINI else OfflineFixtureProvider()
print("provider:", "live Gemini" if LIVE_GEMINI else "offline schema fixture")

pipeline = build_pipeline("paper5", codebook=codebook, provider=provider, config_path=ROOT / "configs" / "paper5.json")


In [ ]:
import tempfile
from dataclasses import replace

from iedi.agents import InterpretationAgent, MultiAgentRuntime, TrustAgent, UXAgent
from iedi.audio import InputAgent, LibrosaAcousticExtractor, PyannoteDiarizer, WhisperASR
from iedi.provenance import InMemoryChain, InMemoryIPFS
from iedi.schemas import FeedbackAction, FeedbackEvent
from iedi.trust import (
    HashChainAuditLog, LocalAppendOnlyEntryStore,
    StaticValidatorAuthorizer, TrustGate, interpretation_result_sha256,
)

# Ephemeral fixtures exercise state transitions without claiming public persistence.
runtime_tmp = tempfile.TemporaryDirectory()
runtime_dir = Path(runtime_tmp.name)
trust_gate = TrustGate.for_pipeline(
    pipeline,
    store=LocalAppendOnlyEntryStore(
        runtime_dir / "entries.jsonl", baseline_entries=pipeline.codebook.entries
    ),
    audit_log=HashChainAuditLog(runtime_dir / "audit.jsonl"),
    authorizer=StaticValidatorAuthorizer({"demo-validator"}),
    ipfs=InMemoryIPFS(),
    chain=InMemoryChain(),
)
runtime = MultiAgentRuntime(
    input_agent=InputAgent(
        asr=WhisperASR(), diarizer=PyannoteDiarizer(),
        acoustic_extractor=LibrosaAcousticExtractor(), require_diarization=True,
    ),
    interpretation_agent=InterpretationAgent(pipeline),
    trust_agent=TrustAgent(trust_gate),
    ux_agent=UXAgent(),
)


In [ ]:
async def run_text_feedback_demo():
    # No acoustic evidence is fabricated: Paper 5 correctly marks review required.
    request = InterpretationRequest(
        utterance="wahala", active_persona_ids=("ng-en-v1",),
        request_id="paper5-demo-request",
    )
    async with runtime:
        presented = await runtime.interpret(request)
        original = pipeline.codebook.get_entry("ng-wahala-1")
        correction = replace(
            original, entry_id="ng-wahala-2-demo", version=2,
            supersedes_entry_id=original.entry_id,
            universal_gloss="human-reviewed demonstration correction",
            reviewed_by=("demo-validator",),
            source_type="offline-demonstration",
            created_at="2026-08-07T00:00:00+00:00",
        )
        event = FeedbackEvent(
            request_id=request.request_id, action=FeedbackAction.ACCEPT,
            actor_id="demo-validator", candidate=presented.result.candidates[0],
            corrected_entry=correction,
            source_result_sha256=interpretation_result_sha256(presented.result),
        )
        receipt = await runtime.submit_feedback(event)
        updated = await runtime.interpret(request)
    return presented, receipt, updated

before, receipt, after = await run_text_feedback_demo()
{
    "route": before.view["route"],
    "missing_affect_requires_review": before.result.needs_human_review,
    "feedback_state": receipt.state.value,
    "indexed_version": receipt.indexed_dataset_version,
    "updated_entry": after.result.candidates[0].entry_id,
}


For real audio, install the audio/diarization extras and call `runtime.interpret_audio(...)`; that path executes ASR → diarization → acoustic extraction → interpretation → UX. In-memory provenance and a static allow-list prove state-machine behavior only. They are not evidence of IPFS durability, blockchain finality, authenticated community governance, or poisoning prevention.
